<a href="https://colab.research.google.com/github/ThodupunooriSaiManish/Deep_Learning/blob/main/DL(Assignment_1_%26_2)_205.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Basic Details:


Dataset Name: Street View House Numbers (SVHN)

Image Size: 32×32

Channels: RGB (3 channels)

Classes: 10 digits (0–9)

Dataset Size: around 600,000 images

#Unit 1

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [3]:
# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Dataset
train_dataset = torchvision.datasets.SVHN(
    root='./data', split='train', download=True, transform=transform)

test_dataset = torchvision.datasets.SVHN(
    root='./data', split='test', download=True, transform=transform)

# Loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [4]:
# Defining MLP Model (Unit-I Core)
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()

        # Input: 3*32*32 = 3072
        self.fc1 = nn.Linear(3*32*32, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)  # 10 classes

    def forward(self, x):
        # Flatten the image
        x = x.view(x.size(0), -1)

        # Hidden layers
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # Output layer
        x = self.fc3(x)

        return x

Flattening:
3×32×32 → 3072 input neurons

Hidden Layers:
512 neurons → learns complex features
256 neurons → refines features

Output Layer:
10 neurons → digits (0–9)

Activation:
Using ReLU (we’ll compare later with Sigmoid)

MLP treats image as a flat vector, ignoring spatial structure. Works, but not optimal for images (CNN will perform better later). ReLU helps in faster convergence compared to sigmoid.

In [5]:
# Loss & Optimizer (Gradient Descent)
# Initialize model
model = MLP()

# Loss Function (for multi-class classification)
criterion = nn.CrossEntropyLoss()

# Optimizer (Basic Gradient Descent)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

CrossEntropyLoss is best suited for multi-class classification.

SGD is simple but:
May converge slowly,
Can oscillate during training

Learning rate plays a critical role:
Too high → unstable training,
Too low → slow learning

In [6]:
# Training the model

epochs = 5

for epoch in range(epochs):
    running_loss = 0.0

    for images, labels in train_loader:

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

Epoch [1/5], Loss: 2502.3183
Epoch [2/5], Loss: 2034.0335
Epoch [3/5], Loss: 1527.8824
Epoch [4/5], Loss: 1262.6340
Epoch [5/5], Loss: 1104.5646


Step-by-step flow:

Forward Pass
Input → Model → Predictions

Loss Calculation
Compare predictions with actual labels

Backpropagation
Compute gradients of loss

Weight Update
Adjust weights using SGD

Loss should decrease over epochs → indicates learning

If loss:

 Not decreasing → learning issue,
 Increasing → learning rate too high

Backpropagation helps:
Efficient weight updates,
Faster convergence

In [7]:
# Testing the model

correct = 0
total = 0

# No gradient needed during testing
with torch.no_grad():
    for images, labels in test_loader:

        outputs = model(images)

        # Get predicted class
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Accuracy
accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 70.10%


torch.no_grad() → disables gradient computation (faster testing)

torch.max() → picks class with highest probability

Compare predictions with actual labels

MLP Performance on SVHN

MLP achieved moderate accuracy (~60–75%)

Performance is limited because:
It flattens images, losing spatial information

Effect of Gradient Descent

SGD successfully reduced loss over epochs

Training was:
Slightly slow,
Sensitive to learning rate

Activation Function (ReLU) Helped in:
Faster convergence,
Avoiding vanishing gradient problem

#Unit 2

In [9]:
# Common training fucntion for all types of GD
def train_model(model, optimizer, epochs=5):
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        running_loss = 0.0

        for images, labels in train_loader:
            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

In [10]:
# Commom test function for all types of GD
def test_model(model):
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Accuracy: {accuracy:.2f}%")

In [20]:
# BGD
model = MLP()

train_loader_full = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(5):
    for images, labels in train_loader_full:

        optimizer.zero_grad()
        outputs = model(images)
        loss = nn.CrossEntropyLoss()(outputs, labels)

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 2.3023
Epoch 2, Loss: 2.3015
Epoch 3, Loss: 2.3008
Epoch 4, Loss: 2.3001
Epoch 5, Loss: 2.2994


In [11]:
#MSGD basic
model = MLP()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

train_model(model, optimizer)
test_model(model)

Epoch [1/5], Loss: 2532.6534
Epoch [2/5], Loss: 2143.7303
Epoch [3/5], Loss: 1591.2608
Epoch [4/5], Loss: 1295.2251
Epoch [5/5], Loss: 1126.2053
Accuracy: 69.81%


In [21]:
#SGD
train_loader_sgd = DataLoader(train_dataset, batch_size=1, shuffle=True)

model = MLP()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

train_model(model, optimizer)
test_model(model)

Epoch [1/5], Loss: 2521.7111
Epoch [2/5], Loss: 2116.9553
Epoch [3/5], Loss: 1594.6121
Epoch [4/5], Loss: 1298.3810
Epoch [5/5], Loss: 1118.1255
Accuracy: 69.69%


In [12]:
# Momentum GD
model = MLP()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

train_model(model, optimizer)
test_model(model)

Epoch [1/5], Loss: 1585.5834
Epoch [2/5], Loss: 940.3898
Epoch [3/5], Loss: 771.0394
Epoch [4/5], Loss: 679.3880
Epoch [5/5], Loss: 616.1668
Accuracy: 80.46%


In [13]:
# Nesterov Accelerated GD
model = MLP()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)

train_model(model, optimizer)
test_model(model)

Epoch [1/5], Loss: 1549.2717
Epoch [2/5], Loss: 897.1996
Epoch [3/5], Loss: 737.8934
Epoch [4/5], Loss: 650.7309
Epoch [5/5], Loss: 590.2853
Accuracy: 80.37%


In [14]:
# AdaGrad
model = MLP()
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01)

train_model(model, optimizer)
test_model(model)

Epoch [1/5], Loss: 1407.8429
Epoch [2/5], Loss: 897.8627
Epoch [3/5], Loss: 765.8872
Epoch [4/5], Loss: 686.8553
Epoch [5/5], Loss: 630.3656
Accuracy: 79.79%


In [15]:
# RMSProp
model = MLP()
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)

train_model(model, optimizer)
test_model(model)

Epoch [1/5], Loss: 1417.3409
Epoch [2/5], Loss: 956.9614
Epoch [3/5], Loss: 824.3570
Epoch [4/5], Loss: 744.4185
Epoch [5/5], Loss: 693.2081
Accuracy: 74.97%


In [18]:
# Adam
model = MLP()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

train_model(model, optimizer)
test_model(model)

Epoch [1/5], Loss: 1556.5026
Epoch [2/5], Loss: 996.7572
Epoch [3/5], Loss: 837.9993
Epoch [4/5], Loss: 745.0172
Epoch [5/5], Loss: 677.1581
Accuracy: 79.24%


Performance Comparison

1)SGD:
Slow convergence: Oscillations in loss

2)Momentum: Faster than SGD, Reduces oscillations

3)Nesterov: Slightly better than Momentum, More accurate updates

Adaptive Methods

4)AdaGrad: Learning rate keeps decreasing, Stops learning early → poor performance

5)RMSProp: Fixes AdaGrad problem, Stable and faster

Best Optimizer → Adam

Combines: Momentum + RMSProp

Advantages: Fast convergence, High accuracy, Stable training

FINAL CONCLUSION

Adam performs best on SVHN dataset

Why?
Adaptive learning rates,
Momentum-based updates,
Handles noisy gradients efficiently.